# Notebook Task — Join Customers and Installments

**Objective:** Join staged installments to customer details, save the joined Delta table, and publish a duplicate flag for the workflow.

## Why?

**For our project:** Add customer names and states to installment records.

**For Data Engineering:** Demonstrate a task that depends on two inputs and passes a data quality result to a downstream condition.

#### 01 — Configure Notebook Parameters

Read the catalog and schema parameters. Keep duplicate simulation disabled for the initial run; we will use it later to test the conditional branch.

In [ ]:
# Define defaults for manual execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")
dbutils.widgets.text("simulate_duplicate", "false")

# Read the current parameter values.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

simulate_duplicate = (
    dbutils.widgets.get("simulate_duplicate").strip().lower()
)
assert simulate_duplicate in {"true", "false"}

#### 02 — Join Installments and Check for Duplicates

Join installments to customer details using `customer_id` and save the result as a Delta table.

Compare the total row count with the distinct full-row count and publish the Boolean result as `has_duplicates` for the downstream If/else task.

In [ ]:
# Enrich installments with customer details.
joined_df = spark.sql(f"""
    SELECT
        i.installment_id,
        i.application_id,
        c.customer_id,
        c.customer_name,
        c.state,
        i.due_date,
        i.amount_due,
        i.payment_status
    FROM {namespace}.installments_staging AS i
    INNER JOIN {namespace}.customers_staging AS c
        ON CAST(i.customer_id AS BIGINT) = c.customer_id
""")

# Add one identical row only during the controlled duplicate test.
if simulate_duplicate == "true":
    joined_df = joined_df.unionByName(joined_df.limit(1))

# Persist the joined records.
target_table = f"{namespace}.customers_installments_silver"

joined_df.write.format("delta").mode("overwrite").saveAsTable(
    target_table
)

# Check for identical full-row duplicates.
stored_df = spark.table(target_table)
total_count = stored_df.count()
distinct_count = stored_df.dropDuplicates().count()
has_duplicates = total_count > distinct_count

# Publish the result for downstream Job tasks.
dbutils.jobs.taskValues.set(
    key="has_duplicates",
    value=has_duplicates
)

print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Has duplicates: {has_duplicates}")

display(stored_df.orderBy("installment_id"))

Total rows: 6
Distinct rows: 6
Has duplicates: False


installment_id,application_id,customer_id,customer_name,state,due_date,amount_due,payment_status
3001,2001,101,Ana Ribeiro,PE,2026-10-20,1750.00,PAID
3002,2001,101,Ana Ribeiro,PE,2026-11-20,1750.00,PENDING
3003,2004,104,Pedro Alves,MG,2026-10-25,4000.00,PAID
3004,2004,104,Pedro Alves,MG,2026-11-25,4000.00,PENDING
3005,2004,104,Pedro Alves,MG,2026-12-25,4000.00,PENDING
3006,2005,105,Julia Santos,BA,2026-11-10,2600.00,PENDING
